In [1]:
# 5_synthetic_population.ipynb
#
# Builds the synthetic population parquet by combining three sources:
#
#   1. Load sipher_optimized.pkl            (synthetic_zone, pidp)
#   2. Map each LSOA (synthetic_zone) to MSOA and Local Authority
#   3. Join feature-engineered UKHLS rows   (o_indresp_feature_eng.pkl) on pidp
#   4. Stream-write to parquet in chunks    (avoids holding full dataset in RAM)

import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from data_pipeline.config_paths import DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES

# ── Config ────────────────────────────────────────────────────────────────────
SIPHER_PKL  = f"../{DATA_FOLDER}/1_pickle_sipher/sipher_optimized.pkl"
FEATURE_PKL = f"../{DATA_FOLDER}/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"
GEO_CSV     = f"../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv"
OUTPUT_FILE = Path(f"../{DATA_FOLDER}/5_synthetic_population/synthetic_population.parquet")
CHUNK_SIZE  = 500_000   # rows of sipher data processed per iteration

for p in [SIPHER_PKL, FEATURE_PKL, GEO_CSV]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} not found — check pipeline prerequisites.")

# ── Load lookup tables (small — kept in memory throughout) ───────────────────
print("Loading lookup tables ...")

geo_cols = ['lsoa21cd', 'msoa21cd', 'msoa21nm', 'ladcd', 'ladnm']
df_geo = (
    pd.read_csv(GEO_CSV, usecols=geo_cols, dtype=str, encoding='latin-1')
    .drop_duplicates(subset='lsoa21cd')
    .set_index('lsoa21cd')
)
print(f"  Geography:  {len(df_geo):,} unique LSOAs")

df_features = pd.read_pickle(FEATURE_PKL)
df_features['pidp'] = df_features['pidp'].astype('int64')
df_features = df_features.set_index('pidp')
print(f"  Features:   {len(df_features):,} UKHLS respondents × {len(df_features.columns)} cols")

# ── Load sipher index only (to determine chunks) ──────────────────────────────
print("\nLoading sipher pickle ...")
df_sipher = pd.read_pickle(SIPHER_PKL)[['synthetic_zone', 'pidp']].copy()
df_sipher['pidp'] = df_sipher['pidp'].astype('int64')
df_sipher.sort_values('synthetic_zone', inplace=True)
df_sipher.reset_index(drop=True, inplace=True)
n_total = len(df_sipher)
n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"  {n_total:,} rows — processing in {n_chunks} chunk(s) of {CHUNK_SIZE:,}")

# ── Stream-write parquet chunk by chunk ───────────────────────────────────────
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
writer = None
n_no_geo = 0
n_unmatched = 0

for i in range(n_chunks):
    start = i * CHUNK_SIZE
    end   = min(start + CHUNK_SIZE, n_total)
    chunk = df_sipher.iloc[start:end].copy()

    # 2. Map LSOA → MSOA / LA
    chunk = chunk.join(df_geo, on='synthetic_zone', how='left')
    n_no_geo += int(chunk['ladcd'].isna().sum())

    # 3. Join UKHLS features
    chunk = chunk.join(df_features, on='pidp', how='left')
    n_unmatched += int(chunk.iloc[:, -1].isna().sum())

    if USE_FOUR_LA_SUBSET:
        chunk = chunk[chunk['ladcd'].isin(FOUR_LA_CODES)]
    if len(chunk) == 0:
        print(f"  Chunk {i+1}/{n_chunks}: no rows in selected LAs — skip")
        continue

    # Write chunk
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_FILE, table.schema, compression='snappy')
    writer.write_table(table)

    print(f"  Chunk {i+1}/{n_chunks}: rows {start:,}–{end:,} written")
    del chunk, table
    gc.collect()

if writer:
    writer.close()

del df_sipher, df_geo, df_features
gc.collect()

# ── Summary ───────────────────────────────────────────────────────────────────
size_mb = OUTPUT_FILE.stat().st_size / 1e6
if n_no_geo:
    print(f"\nWarning: {n_no_geo:,} rows ({100*n_no_geo/n_total:.2f}%) could not be mapped to a geography")
if n_unmatched:
    print(f"Warning: {n_unmatched:,} rows ({100*n_unmatched/n_total:.2f}%) unmatched to a UKHLS respondent")

print(f"\nDone.")
print(f"  Rows:  {n_total:,}")
print(f"  Disk:  {size_mb:.0f} MB")
print(f"  File:  {OUTPUT_FILE}")


Loading lookup tables ...
  Geography:  43,501 unique LSOAs
  Features:   32,849 UKHLS respondents × 64 cols

Loading sipher pickle ...
  52,853,971 rows — processing in 106 chunk(s) of 500,000
  Chunk 1/106: rows 0–500,000 written
  Chunk 2/106: rows 500,000–1,000,000 written
  Chunk 3/106: rows 1,000,000–1,500,000 written
  Chunk 4/106: rows 1,500,000–2,000,000 written
  Chunk 5/106: rows 2,000,000–2,500,000 written
  Chunk 6/106: rows 2,500,000–3,000,000 written
  Chunk 7/106: rows 3,000,000–3,500,000 written
  Chunk 8/106: rows 3,500,000–4,000,000 written
  Chunk 9/106: rows 4,000,000–4,500,000 written
  Chunk 10/106: rows 4,500,000–5,000,000 written
  Chunk 11/106: rows 5,000,000–5,500,000 written
  Chunk 12/106: rows 5,500,000–6,000,000 written
  Chunk 13/106: rows 6,000,000–6,500,000 written
  Chunk 14/106: rows 6,500,000–7,000,000 written
  Chunk 15/106: rows 7,000,000–7,500,000 written
  Chunk 16/106: rows 7,500,000–8,000,000 written
  Chunk 17/106: rows 8,000,000–8,500,000 wr